# L4a: Trading Rules Using Binomial Lattice Models
In this lecture, we turn a lattice price forecast into a precise terminal trading rule. We specify the dated cash flows, a benchmark return, a holding period, and a target before computing a probability.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
> 
> * **Specify and evaluate a terminal trade rule:** Identify the entry cash flow, terminal sale cash flow, holding period, benchmark growth rate, and target return that define an NPV-based rule. Convert the strict return target into a minimum up-move count and evaluate the corresponding binomial tail.
> * **Distinguish terminal and monitored exits:** Explain why a take-profit or stop-loss rule checked during the holding period is a first-passage problem.
> * **Extend the binomial lattice to an N-ary lattice:** Represent each node by branch counts and compute its price and probability.

We begin with a short review of the market setting and the binomial lattice.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb). Estimate a binomial lattice from data, convert a strict fractional-return target into an up-move threshold, and evaluate the binomial tail.

The second example moves beyond two branches per time step:

> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Aray-Lattice-Fall-2026.ipynb). Discretize an empirical one-step growth distribution into several branches, build a recombining lattice, and inspect terminal probability mass.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: Exchanges, Stocks, and the Binomial Lattice Model
Last week, we introduced equities, exchanges, orders, and the binomial lattice model. Here is the market setting we need today.

* __What is a stock?__ An ownership share in a company. Check out [this video](https://www.youtube.com/watch?v=XH4oPjaPlkk) for a quick overview.
* __What is an exchange?__ A regulated marketplace where securities are bought and sold. U.S. securities trade on several registered exchanges, including the New York Stock Exchange and Nasdaq. See the [SEC list of national securities exchanges](https://www.sec.gov/about/divisions-offices/division-trading-markets/national-securities-exchanges).
* __What is an exchange-traded fund (ETF)?__ A type of investment fund (a basket of assets) that is traded on stock exchanges, much like stocks. ETFs hold collections of assets such as stocks, commodities, or bonds. Check out [this video](https://www.youtube.com/watch?v=Tv4pkivGvdU) for a quick overview.
* __What is a mutual fund?__ An investment vehicle that pools money from many investors to purchase a diversified portfolio of stocks, bonds, or other securities. Mutual funds are managed by professional portfolio managers and are priced once a day. Check out [this video](https://www.youtube.com/watch?v=HkP7Yk1bX3I) for a quick overview.
* __Order types:__ A market order seeks immediate execution, but its price is not guaranteed. A limit order specifies a price or better, but execution is not guaranteed. See the [SEC overview of order types](https://www.investor.gov/introduction-investing/investing-basics/how-stock-markets-work/types-orders).

<div>
    <center>
        <img src="figs/Fig-Exchage-Schematic.svg" width="800"/>
    </center>
</div>

### Company Profile: Citadel Securities
[Citadel Securities](https://www.citadelsecurities.com) is a market maker. A market maker posts prices at which it is willing to buy and sell, helping other participants trade. The bid--ask spread is one source of compensation for providing this liquidity.

### The Binomial Lattice Model
A binomial lattice model is a discrete-time model used to represent the evolution of an asset's price over time.
> __Binomial Lattice Model__: A binomial lattice model assumes that at each (future) time step, the asset can either move up by a factor $u$ with probability $p$ or down by a factor $d$ with probability $(1-p)$, creating a tree-like structure of possible future prices. At time step $t\geq{0}$, there will be $t+1$ possible prices. 

<div>
    <center>
        <img src="figs/Fig-Lattice-Schematic.svg" width="800" alt="Three panels showing 1-, 2-, and 3-step-ahead binomial lattices: from the current price, each step branches up with probability p and factor u or down with probability 1 minus p and factor d, and the branches recombine into t plus 1 nodes at time t"/>
    </center>
</div>

The tree stays this compact because the model holds its parameters fixed across the lattice.

> __Key assumptions__: The probability $p$ and the up and down factors are independent of the time step $t$, i.e., these parameters are constant across the lattice. The up, down, and probability parameters are specific to each firm.

The complete price distribution at level $t$ follows:
$$
\boxed{
S_t = S_0 u^k d^{t-k} \quad \text{with probability} \quad \binom{t}{k}\;p^k (1-p)^{t-k}, \quad k = 0,1,\ldots,t
}
$$

The constant-parameter binomial model does not capture features such as heavy tails or volatility clustering.

It remains useful because it gives a transparent probability distribution over a finite set of future states. We will later use related lattices for derivatives pricing.
___

## Net Present Value (NPV) Trade Rule
We now turn a terminal price model into a scheduled long-position rule.

> __Scenario:__ Suppose we purchase $n_{0}$ shares of ticker `XYZ` at time $t=0$ (today) for $S_{0}$ USD/share.
> Then, sometime later, at $T=N\Delta{t}$, we sell all $n_{0}$ shares at a share price of $S_{T}$ USD/share, where $N\geq{0}$ is the number of time steps (integer) and $\Delta{t}$ is the time step (e.g., one trading day written in units of years).

This is a __long position__: its value increases when the share price rises. The per-share profit is a line through the purchase price, with a breakeven point at entry, a profit region above it, and a loss region below it.

<div>
    <center>
        <img src="figs/Fig-TradeRule-Schematic.svg" width="800" alt="Per-share profit of a long position in ticker XYZ versus terminal share price: a straight line crossing zero profit at the benchmark-adjusted terminal breakeven price B, with a profit point P above it and a loss point L below it, asking the probability of reaching each at the exit time"/>
    </center>
</div>

However, $S_T>S_0$ alone does not guarantee a positive benchmark-adjusted NPV after the time value of money and trading costs. In a frictionless baseline, the position has two cash flows: the purchase at $t=0$ and the sale at $t=T$.
$$
\begin{align*}
\texttt{NPV}(g_b, T) &= \underbrace{-n_{0}\;{S_{0}}}_{\text{entry (now)}} + \underbrace{n_{0}\;{S_{T}}\;\mathcal{D}_{T,0}^{-1}(g_b)}_{\text{exit (future)}}\quad\Longrightarrow\text{divide by initial investment}\;n_{0}\;{S_{0}}\\
\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}} &= \frac{S_{T}\;\mathcal{D}_{T,0}^{-1}(g_b) - S_{0}}{S_{0}}\\
\underbrace{\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}}}_{\text{fractional return}} &= \left(\frac{S_{T}}{S_{0}}\right)\;\mathcal{D}_{T,0}^{-1}(g_b) - 1\quad\blacksquare
\end{align*}
$$
Here $g_b$ is the continuously compounded annual benchmark growth rate, consistent with L1b. Thus $\mathcal{D}_{T,0}(g_b)=e^{g_bT}$ is the accumulation factor and $\mathcal{D}^{-1}_{T,0}(g_b)=e^{-g_bT}$ discounts the terminal sale value to time 0. The matching one-step gross benchmark return from L3b is $R_b=e^{g_b\Delta t}$.

> __Interpretation__: The left-hand side is a dimensionless present-value return on the initial share cost. A probability model for $S_T$ therefore gives a model probability for any specified terminal return target.


Let's take a closer look at this NPV expression, in the context of short versus long holding periods, and our binomial lattice model for the share price $S_{T}$.

### Short holding period
When $|g_b|T$ is small, $\mathcal{D}_{T,0}(g_b)\approx1$. The present-value return is then approximately:
$$
\begin{align*}
\frac{\texttt{NPV}(T)}{n_{0}\;{S_{0}}} &\approx \frac{S_{T}}{S_{0}} - 1\quad\blacksquare
\end{align*}
$$
Thus, over a short horizon, the scaled NPV is approximately the fractional change in share price. A model for $S_T$ then gives the distribution of this quantity.

Given the randomness of the share price $S_{T}$, we can use our binomial lattice model to compute the distribution of potential future share prices $S_{T}$ at time $T=N\Delta{t}$, and the probability of achieving a desired fractional return (ROI).

At the end of the holding period $T=N\Delta{t}$, the scaled share price $S_{T}/S_{0}$ will be one of $N+1$ possible values:
$$
\boxed{
\frac{S_{T}}{S_{0}} = u^k d^{N-k} \quad \text{with probability} \quad \binom{N}{k}\;p^k (1-p)^{N-k}, \quad k = 0,1,\ldots,N
}
$$
where $u>1$ is the up-factor, $0<d<1$ is the down-factor, and $0<p<1$ is the probability of an up-move. Thus, the scaled NPV (fractional return) at time $T=N\Delta{t}$ will be one of $N+1$ possible values:
$$
\frac{\texttt{NPV}(T)}{n_{0}\;{S_{0}}} \approx u^k d^{N-k} - 1 \quad \text{with probability} \quad \binom{N}{k}\;p^k (1-p)^{N-k}, \quad k = 0,1,\ldots,N
$$


### Long holding period
For a longer horizon, or whenever $|g_b|T$ is not small, retain the discount factor:
$$
\begin{align*}
\texttt{NPV}(g_b, T) &= n_{0}\;(S_{T}\;\mathcal{D}_{T,0}^{-1}(g_b) - S_{0})\quad\Longrightarrow\text{divide by initial investment}\;n_{0}\;{S_{0}}\\
\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}} &= \frac{S_{T}\;\mathcal{D}_{T,0}^{-1}(g_b) - S_{0}}{S_{0}}\\
\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}} &= \left(\frac{S_{T}}{S_{0}}\right)\;\mathcal{D}_{T,0}^{-1}(g_b) - 1\quad\blacksquare
\end{align*}
$$
We can again use our binomial lattice model to compute the distribution of potential future share prices $S_{T}$ at time $T=N\Delta{t}$ and the probability of achieving a desired fractional return (ROI). Substituting the binomial lattice model for $S_{T}$ into the scaled NPV expression gives:
$$
\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}} = \frac{u^k d^{N-k}}{\mathcal{D}_{T,0}(g_b)} - 1 \quad \text{with probability} \quad \binom{N}{k}\;p^k (1-p)^{N-k}, \quad k = 0,1,\ldots,N
$$
where $u>1$ is the up-factor, $0<d<1$ is the down-factor, $0<p<1$ is the probability of an up-move, and $\mathcal{D}_{T,0}^{-1}(g_b)=e^{-g_bT}$ is the time-0 discount factor.

___

## Cumulative Probability of a Trade Exit
A terminal rule usually asks whether the return clears a target, not whether it equals one particular node value. We therefore need a cumulative probability.

> __Strict terminal target__: For a holding period $T=N\Delta t$, compute $\mathbb{P}\left({\texttt{NPV}(g_b,T)}/{(n_0S_0)}>\rho_\star\right)$. The event is strict: a node exactly equal to $\rho_\star$ is not counted.

Think of $k$ (the number of up-moves) as the only random thing. The payoff ratio is given by:
$$
\rho(k)=\frac{\texttt{NPV}(g_b, T)}{n_{0}S_{0}}
=\frac{u^{k}d^{\,N-k}}{\mathcal D_{T,0}(g_b)}-1,
\quad
\mathcal D_{T,0}(g_b)=e^{g_b\,N\,\Delta t}.
$$
This is monotonically **increasing** in $k$ whenever $u>d>0$. Therefore, the event $\rho(k)>\rho_{\star}$ is equivalent to $k$ being above a threshold.
Start by solving $u^{k}d^{N-k}/\mathcal D_{T,0}(g_b)-1>\rho_{\star}$ for $k$:
$$
\frac{u^{k}d^{\,N-k}}{\mathcal D_{T,0}(g_b)}-1>\rho_\star
\;\Longleftrightarrow\;
u^{k}d^{\,N-k}>(1+\rho_\star)\,\mathcal D_{T,0}(g_b).
$$
Take the natural logarithm of both sides and solve for $k$. Let $A=\ln(u/d)>0$ and $B=\ln d<0$:
$$
k\ln u+(N-k)\ln d>\ln(1+\rho_\star)+g_b\,N\,\Delta t
\;\Longleftrightarrow\;
kA+NB>\ln(1+\rho_\star)+g_b\,N\,\Delta t.
$$
Define $\tau(\rho_{\star})$ as the right-hand side after isolating $k$:
$$
\tau(\rho_{\star}) = \frac{\ln(1+\rho_\star)+g_b\,N\,\Delta t - N\ln d}{\ln(u/d)}.
$$
Then the smallest integer that clears the $\rho>\rho_{\star}$ bar is:
$$
\boxed{
k_{\min} = \lfloor \tau(\rho_{\star}) \rfloor + 1
}.
$$
With the up-move count $K_N\sim\texttt{Binomial}(N,p)$, the cumulative probability is the binomial tail:
$$
\boxed{
\mathbb P\!\left(\frac{\texttt{NPV}(g_b,T)}{n_{0}S_{0}}>\rho_\star\right)
=\sum_{k=\max(0,k_{\min})}^{N}\binom{N}{k}p^{k}(1-p)^{\,N-k}
=1-\sum_{k=0}^{k_{\min}-1}\binom{N}{k}p^{k}(1-p)^{\,N-k}.
}
$$
This is a closed-form expression for the model probability of strictly exceeding the terminal target. The threshold can lie outside the feasible set of up-move counts, so we must handle the edge cases explicitly.

> __Edge cases__
>
> * If $k_{\min}\le 0$, probability is $1$.
> * If $k_{\min}>N$, probability is $0$ (the hurdle is too high).
> * At fixed $N$, increasing the benchmark growth rate $g_b$ raises the threshold and cannot increase the tail probability.
> * Changing the horizon changes both the threshold and the binomial distribution because $N$ changes. The full probability must be recomputed; there is no general monotonic conclusion from the discount factor alone.

The complementary event describes terminal returns that do not clear the strict target.

The probability of the return being less than or equal to a specified $\rho_{\star}$ is simply the complement of the probability we just derived:
$$
\boxed{
\mathbb P\!\left(\frac{\texttt{NPV}(g_b,T)}{n_{0}S_{0}}\leq \rho_\star\right)
=1 - \mathbb P\!\left(\frac{\texttt{NPV}(g_b,T)}{n_{0}S_{0}}> \rho_\star\right).
}
$$

Let's look at an example.

> __Example__
>
> [▶ Explore a terminal target probability](CHEME-5660-L4a-Example-CumulativeProbabilityLattice-Fall-2026.ipynb). The notebook estimates a real-world binomial lattice, computes the strict threshold, and evaluates the analytic tail with explicit edge-case handling.
___

## Terminal Rules and First-Passage Rules Are Different
The binomial-tail formula above concerns liquidation at the scheduled date $T$. A take-profit or stop-loss rule monitored at every step depends on the first time a boundary is crossed. It is therefore path dependent.

> __Why terminal probabilities are not enough__: Two price paths can end at the same terminal node but cross a trading boundary at different earlier times. A monitored rule can execute differently along those paths even though their terminal prices match.

To evaluate a monitored rule, propagate only the probability mass that has not already crossed a boundary, and record the mass absorbed at each first crossing. The execution rule also matters: a stop order becomes a market order after its trigger and need not execute at the trigger price; a limit order specifies a price but may not fill.

The remainder of this lecture returns to terminal distributions.
___

## N-Ary Lattice Models
A binomial lattice keeps only two one-step outcomes. An N-ary lattice retains $m$ outcomes at each step.

> __Idea__: A trinomial lattice might use up, middle, and down factors. More generally, choose positive factors $f_1,\ldots,f_m$ with probabilities $p_1,\ldots,p_m$ that sum to one.

We use a __recombining tree__: order does not matter when the branch factors are constant, so paths with the same branch-count vector share one state.

The number of unique nodes at level $t$ is given by the stars-and-bars count
$$
L_t = \binom{t + m - 1}{m - 1}.
$$
Summing over levels $r = 0,\dots,h$ gives the total number of unique nodes in a height-$h$ tree:
$$
\sum_{r=0}^{h} L_r = \binom{h + m}{m} = \binom{h + m}{h}.
$$
When storing nodes in level order in a flat array, the offset to the first node of level $t$ is the number of nodes in all prior levels:
$$
O(t) = \sum_{r=0}^{t - 1} L_r = \binom{t + m - 1}{m}.
$$

At each node, we are going to track the number of times each of the $m$ possible outcomes has occurred. Let $\mathbf{x} = (x_1, x_2, \ldots, x_m)$ be a vector where $x_j$ is the number of times outcome $j$ has occurred. The total number of steps taken to reach this node is $t = \sum_{j=1}^{m} x_j$, i.e., the node sits at level $t$. The price at this node can be expressed as:
$$
S_t = S_0 \prod_{j=1}^{m} f_j^{x_j}
$$
where $f_j$ is the factor associated with outcome $j$, e.g., the price change factor. The probability of reaching this node is given by the multinomial distribution:
$$
P(\mathbf{x}) = \left(\frac{t!}{x_1! x_2! \cdots x_m!}\right) p_1^{x_1} p_2^{x_2} \cdots p_m^{x_m}
$$
where $p_j$ is the probability of outcome $j$ at each step. More branches give a finer one-step discretization, but they do not guarantee a better forecast; the parameters still require validation.

> __Example__
>
> [▶ Explore N-ary lattice models](CHEME-5660-L4a-Example-N-Aray-Lattice-Fall-2026.ipynb). The notebook discretizes empirical log growth into three branches, builds the recombining count states, and plots probability mass at a selected level.

This discrete construction prepares us for the continuous stochastic price models introduced in the next lecture.
___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L4b; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ First-passage exit rules](advanced/first-passage/CHEME-5660-L4a-Advanced-FirstPassage-ExitRules-Fall-2026.ipynb). Compute exact take-profit and stop-loss first-passage probabilities by propagating only the probability mass for positions that remain open.
* [▶ Execution-aware probability of profit](advanced/execution/CHEME-5660-L4a-Advanced-ExecutionAware-ProbabilityOfProfit-Fall-2026.ipynb). Add the bid-ask spread, fees, and slippage to the terminal probability-of-profit calculation and compare with the frictionless rule.
___

## Summary
In this lecture, we converted lattice price distributions into precisely defined terminal target events and then extended the state space beyond two branches.

> __Key Takeaways:__
>
> * **A terminal target becomes a binomial tail:** Once the entry price, terminal exit, horizon, benchmark, costs, and strictness fully specify the rule, the monotone terminal return converts the target into a minimum up-move count whose probability is a binomial tail, with infeasible thresholds handled explicitly.
>
> * **Monitored exits are path dependent:** A take-profit or stop-loss rule checked during the holding period is a first-passage problem, not a terminal-node event.
>
> * **N-ary states use branch counts:** A multinomial count vector determines each recombining node's price and probability, and the number of states grows with the branch count and horizon.

Next time: Continuous stochastic models of price dynamics.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products, or any investment or trading advice or strategy, is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___